### Installation

In [26]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

`FastLanguageModel` supports loading nearly any model now! This includes Vision and Text models!

In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/data/MW-Amit/gemma-3-4b/models--google--gemma-3-4b-it/snapshots/093f9f388b31de276ce2de164bdc2081324b9767",
    max_seq_length = 2048,
    dtype = torch.float16,          # or bfloat16
    load_in_4bit = True,             # if using QLoRA
    device_map = {"": torch.cuda.current_device()}  # 🔑 CRITICAL
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/data/MW-Amit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.4: Fast Gemma3 patching. Transformers: 4.56.2. vLLM: 0.11.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.045 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.52s/it]


We now add LoRA adapters so we only need to update a small amount of parameters!

In [5]:
# model = FastModel.get_peft_model(
model = FastLanguageModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients


<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use Our own dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [6]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [8]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/data/MW-Amit/Finetuning Codes/generalized_intents_2400_grouped.jsonl",
    split="train"
)


We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [9]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Let's see how row 100 looks like!

In [10]:
dataset[100]

{'instruction': 'You are an expert intent classifier for a conversational voice bot. Classify the user\'s MOST RECENT response into exactly ONE of these intents: answer (direct or relevant reply to the bot\'s question, e.g., "Yes", "No", numbers, short descriptions), denial (the user refuses to answer, ends the call, or explicitly declines to provide information), repeat (the user asks to repeat or clarify the question), query (the user asks a counter-question or requests more information, e.g., "Why do you need this?"), greeting (only at the very start of the call when the user provides a simple greeting). New rule: If the user responds with a simple "Yes" or "No", you MUST check the conversation context to decide whether that response is a direct answer to the question (label as answer) or is being used to refuse/deny participation (label as denial). Greeting is ONLY valid on the first 1–2 turns. Use the full conversation history provided below to understand context. Return ONLY vali

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`. We remove the `<bos>` token using removeprefix(`'<bos>'`) since we're finetuning. The Processor will add this token before training and the model expects only one.

In [12]:
def formatting_prompts_func(examples):
    texts = []

    for instr, inp, out in zip(
        examples["instruction"],
        examples["input"],
        examples["output"]
    ):
        messages = [
            {"role": "system", "content": instr},
            {"role": "user", "content": inp},
            {"role": "assistant", "content": out},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )

        if text.startswith("<bos>"):
            text = text[len("<bos>"):]

        texts.append(text)

    return {"text": texts}


In [13]:
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset.column_names
)


Let's see how the chat template did! Notice there is no beginning of sequence `<bos>` token as the processor tokenizer will be adding one.

In [14]:
dataset[100]["text"]

'<start_of_turn>user\nYou are an expert intent classifier for a conversational voice bot. Classify the user\'s MOST RECENT response into exactly ONE of these intents: answer (direct or relevant reply to the bot\'s question, e.g., "Yes", "No", numbers, short descriptions), denial (the user refuses to answer, ends the call, or explicitly declines to provide information), repeat (the user asks to repeat or clarify the question), query (the user asks a counter-question or requests more information, e.g., "Why do you need this?"), greeting (only at the very start of the call when the user provides a simple greeting). New rule: If the user responds with a simple "Yes" or "No", you MUST check the conversation context to decide whether that response is a direct answer to the question (label as answer) or is being used to refuse/deny participation (label as denial). Greeting is ONLY valid on the first 1–2 turns. Use the full conversation history provided below to understand context. Return ONLY

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [15]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [16]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Let's verify masking the instruction part is done! Let's print the 100th row again.  Notice how the sample only has a single `<bos>` as expected!

In [17]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<bos><start_of_turn>user\nYou are an expert intent classifier for a conversational voice bot. Classify the user\'s MOST RECENT response into exactly ONE of these intents: answer (direct or relevant reply to the bot\'s question, e.g., "Yes", "No", numbers, short descriptions), denial (the user refuses to answer, ends the call, or explicitly declines to provide information), repeat (the user asks to repeat or clarify the question), query (the user asks a counter-question or requests more information, e.g., "Why do you need this?"), greeting (only at the very start of the call when the user provides a simple greeting). New rule: If the user responds with a simple "Yes" or "No", you MUST check the conversation context to decide whether that response is a direct answer to the question (label as answer) or is being used to refuse/deny participation (label as denial). Greeting is ONLY valid on the first 1–2 turns. Use the full conversation history provided below to understand context. Return

Now let's print the masked out example - you should see only the answer is present:

In [18]:
tokenizer.decode(
    [
        token_id if token_id != -100 else tokenizer.pad_token_id
        for token_id in trainer.train_dataset[100]["labels"]
    ],
    skip_special_tokens=True
)


'{"intent": "correction"}\n'

In [19]:
print("INPUT:")
print(tokenizer.decode(
    trainer.train_dataset[100]["input_ids"],
    skip_special_tokens=True
))

print("\nLABELS:")
print(tokenizer.decode(
    [x if x != -100 else tokenizer.pad_token_id
     for x in trainer.train_dataset[100]["labels"]],
    skip_special_tokens=True
))


INPUT:
user
You are an expert intent classifier for a conversational voice bot. Classify the user's MOST RECENT response into exactly ONE of these intents: answer (direct or relevant reply to the bot's question, e.g., "Yes", "No", numbers, short descriptions), denial (the user refuses to answer, ends the call, or explicitly declines to provide information), repeat (the user asks to repeat or clarify the question), query (the user asks a counter-question or requests more information, e.g., "Why do you need this?"), greeting (only at the very start of the call when the user provides a simple greeting). New rule: If the user responds with a simple "Yes" or "No", you MUST check the conversation context to decide whether that response is a direct answer to the question (label as answer) or is being used to refuse/deny participation (label as denial). Greeting is ONLY valid on the first 1–2 turns. Use the full conversation history provided below to understand context. Return ONLY valid JSON 

In [20]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.045 GB.
4.381 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [21]:
import torch

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))


torch version: 2.9.0+cu128
CUDA available: True
CUDA device count: 1
Current device: 0
Device name: NVIDIA L4


In [22]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,400 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 16,394,240 of 4,316,473,712 (0.38% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.435600
2,3.204500
3,1.795500
4,0.275000
5,1.826800
6,0.957600
7,0.110900
8,0.106600
9,0.004600
10,0.208100


In [23]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

77.5882 seconds used for training.
1.29 minutes used for training.
Peak reserved memory = 4.381 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 19.873 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [ ]:
from unsloth.chat_templates import get_chat_template
import torch

torch._dynamo.disable()  # 🔑 critical for Gemma-3 inference

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",
)

messages = [
    {
        "role": "system",
        "content": [{
            "type": "text",
            "text": (
                "You are an expert intent classifier for a conversational voice bot. "
                "Classify the user's MOST RECENT response into exactly ONE of these intents: "
                "answer, denial, repeat, query, greeting, correction. "
                "Return ONLY valid JSON like {\"intent\": \"answer\"}."
            )
        }]
    },
    {
        "role": "user",
        "content": [{
            "type": "text",
            "text": (
                "Bot: What was the appointment date?\n"
                "User: why do you want to know that?"
            )
        }]
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

# 🔑 Move ALL inputs to GPU
inputs = {k: v.to("cuda") for k, v in inputs.items()}

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        do_sample=False,
    )

prediction = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(prediction)


{"intent": "query"}


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [43]:
model.save_pretrained("gemma-3")  # Local saving
tokenizer.save_pretrained("gemma-3")
# model.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving

['gemma-3/processor_config.json']

In [35]:
# =========================
# Interactive Gemma Intent Classifier (Accelerate-safe)
# =========================

import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------- CONFIG --------
MODEL_PATH = "/data/MW-Amit/Finetuning Codes/gemma-3"
MAX_NEW_TOKENS = 16

SYSTEM_PROMPT = (
    "You are an INTENT CLASSIFIER.\n"
    "Output exactly ONE JSON object and NOTHING ELSE.\n\n"
    "Allowed intents:\n"
    "- answer\n"
    "- denial\n"
    "- repeat\n"
    "- query\n"
    "- greeting\n"
    "- correction\n\n"
    "Format strictly as:\n"
    "{\"intent\":\"<label>\"}"
)

# -------- LOAD TOKENIZER --------
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)

# -------- LOAD MODEL (DO NOT TOUCH PARAMETERS) --------
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
    offload_buffers=True,        # IMPORTANT: avoids dtype/offload issues
    trust_remote_code=True,
)
model.eval()

print("Model loaded successfully.\n")

# -------- SAFE JSON EXTRACTION --------
INTENT_REGEX = re.compile(r'\{\s*"intent"\s*:\s*"([^"]+)"\s*\}')

def predict_intent(user_text: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.0,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    match = INTENT_REGEX.search(text)
    if match:
        return {"intent": match.group(1)}

    return {"intent": "unknown", "raw_output": text}

# -------- INTERACTIVE LOOP --------
print("Intent classifier ready.")
print("Type input and press Enter. Type 'exit' to stop.\n")

while True:
    user_input = input("User  > ").strip()
    if user_input.lower() in {"exit", "quit"}:
        print("System> Exiting.")
        break

    print(f"Input  : {user_input}")
    result = predict_intent(user_input)
    print(f"Output : {json.dumps(result)}")
    print("-" * 60)


Loading tokenizer...
Loading model...


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.84it/s]


Model loaded successfully.

Intent classifier ready.
Type input and press Enter. Type 'exit' to stop.

Input  : what is your pan no.
Output : {"intent": "query"}
------------------------------------------------------------
Input  : whad do you mean?
Output : {"intent": "query"}
------------------------------------------------------------
Input  : i didnt get it?
Output : {"intent": "query"}
------------------------------------------------------------
Input  : pardon!
Output : {"intent": "correction"}
------------------------------------------------------------
Input  : can you ask again?
Output : {"intent": "query"}
------------------------------------------------------------
Input  : repeat
Output : {"intent": "correction"}
------------------------------------------------------------
Input  : sorry i cant share the details with you
Output : {"intent": "denial"}
------------------------------------------------------------
Input  : hello
Output : {"intent": "greeting"}
-----------------